nohup python pred_snap.py --model mistral-7B-instruct-v0.2 --compress_args_path ablation_c4096_w64_b32_k7_maxpool.json > 20251210c4096w64b32.log 2>&1 &
nohup python pred_snap.py --model mistral-7B-instruct-v0.2 --compress_args_path ablation_c4096_w64_b128_k7_maxpool.json > 20251210c4096w64b128.log 2>&1 &
nohup python pred_snap.py --model mistral-7B-instruct-v0.2 --compress_args_path ablation_c4096_w64_b512_k7_maxpool.json > 20251210c4096w64b512.log 2>&1 &
nohup python pred_snap.py --model mistral-7B-instruct-v0.2 --compress_args_path ablation_c2048_w64_b64_k7_maxpool.json > 20251210c2048w64b64.log 2>&1 &



新实验：
nohup python pred_snap.py --model mistral-7B-instruct-v0.2 --compress_args_path ablation_c4096_w32_k7_maxpool.json > 20260420c4096w32.log 2>&1 &
nohup python pred_snap.py --model mistral-7B-instruct-v0.2 --compress_args_path ablation_c2048_w32_k7_maxpool.json > 20260420c2048w32.log 2>&1 &
nohup python pred_snap.py --model mistral-7B-instruct-v0.2 --compress_args_path ablation_c1024_w32_k7_maxpool.json > 20260420c1024w32.log 2>&1 &
nohup python pred_snap.py --model mistral-7B-instruct-v0.2 --compress_args_path ablation_c512_w32_k7_maxpool.json > 20260420c512w32.log 2>&1 &


chmod +x autorun1.sh
nohup ./autorun1.sh > autorun10.nohup.log 2>&1 &

kill -9 xxxxxxx

pkill -f run_seq.sh

先用现在这个版本的VAE测试一下压缩还原能力：
- 即使用latent压缩存储KV、然后在推理时还原KV以计算近似注意力

如果还原能力很好，其实就可以考虑先做成使用latent压缩KV的方案，至于解码峰值这些的后面考虑优化/或者在这上面训练模型然后看预测效果也行，总之不管是预测还是压缩都需要保证压缩还原能力相对好

无补偿性能（对照组）

| 窗口大小 | 最大缓存容量 | 补偿块大小 | 显存消耗  | 推理速度(s) | LongBench得分 |
|------|--------|-------|-------|---------|------------|
| 16   | 4096   | 无补偿   | 20430 | 331     | 33.56      |
| 32   | 512    | 无补偿    | -     | 312     | 28.78      |
| 32   | 1024   | 无补偿    | -     | 312     | 29.57      |
| 32   | 2048   | 无补偿   | 18485 | 311     | 32.44      |
| 32   | 4096   | 无补偿   | 20430 | 336     | 33.71      |
| 64   | 4096   | 无补偿   | 20430 | 331     | 33.05      |

VAE:

| 窗口大小 | 最大缓存容量 | 补偿块大小 | 显存消耗  | 推理速度(s) | LongBench得分 |
|------|--------|-------|-------|---------|------------|
| 32   | 512    | 无补偿    | -     | 312     | 27.79      |
| 32   | 1024   | 无补偿    | -     | 312     | 29.57      |
| 32   | 2048   | 无补偿   | 18485 | 311     | 32.44      |
| 32   | 4096   | 无补偿   | 20430 | 336     | 33.13      |

很好，VAE之后性能几乎不损失
可以考虑先做成使用latent压缩KV的方案（也就是暂时先不考虑拿小模型预测latent），因为VAE很小，latent应该压缩率也相当高（肯定比量化高）
